# Event-TimeRAF Results and Figures

This notebook reads frozen artifacts from the main pipeline. It does not train models or alter predictions.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'configs' / 'default.yaml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from event_timeraf.config import load_config
from event_timeraf.plots import plot_horizon_metrics

cfg = load_config(PROJECT_ROOT / 'configs' / 'default.yaml', PROJECT_ROOT)
metrics = pd.read_csv(cfg.paths.outputs / 'tables' / 'metrics.csv')
predictions = pd.read_parquet(cfg.paths.outputs / 'predictions' / 'predictions.parquet')
explanations = pd.read_parquet(cfg.paths.outputs / 'evidence' / 'explanations.parquet')


In [ ]:
overall = metrics.loc[
    (metrics['horizon'].astype(str) == 'overall') & (metrics['subset'] == 'all')
].sort_values('mse')
display(overall)
overall.to_csv(cfg.paths.outputs / 'tables' / 'main_results.csv', index=False)
plot_horizon_metrics(metrics, 'mse', cfg.paths.outputs / 'figures' / 'mse_by_horizon.png')
plot_horizon_metrics(metrics, 'mae', cfg.paths.outputs / 'figures' / 'mae_by_horizon.png')


In [ ]:
errors = predictions.assign(
    squared_error=(predictions['actual'] - predictions['prediction']) ** 2,
    absolute_error=(predictions['actual'] - predictions['prediction']).abs(),
)
drift_results = (
    errors.groupby(['model', 'drift_flag'], as_index=False)
    .agg(count=('actual', 'size'), mse=('squared_error', 'mean'), mae=('absolute_error', 'mean'))
)
drift_results.to_csv(cfg.paths.outputs / 'tables' / 'drift_period_results.csv', index=False)
event_results = (
    errors.groupby(['model', 'event_flag'], as_index=False)
    .agg(count=('actual', 'size'), mse=('squared_error', 'mean'), mae=('absolute_error', 'mean'))
)
event_results.to_csv(cfg.paths.outputs / 'tables' / 'event_period_results.csv', index=False)
display(drift_results)
display(event_results)
display(explanations.sort_values('drift_score', ascending=False).head(10))
